# SHAP for XGBoost (interpretability)

set CWD + load modeling data

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

import shap
import matplotlib.pyplot as plt

# Ensure we are in project root
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("CWD:", Path.cwd())

PROCESSED = Path("data/processed")
REPORTS = Path("reports")
FIGS = REPORTS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

# Load latest modeling parquet
candidates = sorted(PROCESSED.glob("modeling_*.parquet"), key=lambda p: p.stat().st_mtime, reverse=True)
model_path = candidates[0]
print("Using:", model_path.name)

df = pd.read_parquet(model_path)

target_col = "auc"
y = df[target_col].astype(float).values
X = df.drop(columns=[target_col])

depmap_id = X["depmap_id"].astype(str)
X = X.drop(columns=["depmap_id"])

X.shape, y.shape

c:\Users\josha\Codeacademy Projects\Data Scientist Portfolio Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CWD: c:\Users\josha\Codeacademy Projects\Data Scientist Portfolio Project
Using: modeling_dinaciclib_auc.parquet


((476, 19193), (476,))

sklearn-compatible selector (copy from Notebook 04)

In [2]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class TopVarianceSelector(BaseEstimator, TransformerMixin):
    def __init__(self, top_n=2000):
        self.top_n = int(top_n)

    def fit(self, X, y=None):
        X_arr = X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)
        v = np.var(X_arr, axis=0)
        n = min(self.top_n, X_arr.shape[1])
        self.keep_idx_ = np.argsort(v)[::-1][:n]

        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        else:
            self.feature_names_in_ = None
        return self

    def transform(self, X):
        check_is_fitted(self, "keep_idx_")
        X_arr = X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)
        return X_arr[:, self.keep_idx_]

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "keep_idx_")
        if input_features is None:
            input_features = self.feature_names_in_
        input_features = np.asarray(input_features, dtype=object)
        return input_features[self.keep_idx_]

rebuild preprocessing + train a small XGBoost (fast + SHAP-ready)

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("selector", TopVarianceSelector(top_n=2000)),
])

X_p = prep.fit_transform(X)
feature_names = prep.named_steps["selector"].get_feature_names_out(X.columns)

# small split just for SHAP demo
X_tr, X_te, y_tr, y_te = train_test_split(X_p, y, test_size=0.2, random_state=42)

xgb = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
)

xgb.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
getattr(xgb, "best_iteration", None)

68

SHAP summary plot (sampled for speed)

In [4]:
rng = np.random.default_rng(42)
n = min(400, X_te.shape[0])
idx = rng.choice(X_te.shape[0], size=n, replace=False)

X_shap = X_te[idx]
X_shap_df = pd.DataFrame(X_shap, columns=feature_names)

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_shap_df)

plt.figure()
shap.summary_plot(shap_values, X_shap_df, show=False)
out = FIGS / f"shap_summary_{model_path.stem}.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.close()
print("Wrote:", out)

Wrote: reports\figures\shap_summary_modeling_dinaciclib_auc.png
